## 문서 형식 확장

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day04" else here
os.chdir(ROOT)

SANDBOX = ROOT / "sandbox" / "w5" / "day04"        

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
MODELS = ROOT.parent / "models"         
FORMATS = SAMPLES / "_formats"


프로젝트 루트  : C:\workspace\hanwha-agent


In [2]:
if not SAMPLES.is_dir():
    print('재료 폴더가 없습니다:', SAMPLES)
    print('9/17 노트북 00 번의 복사 셀을 한 번 더 돌리세요.')
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]
    root_files = [p for p in SAMPLES.iterdir() if p.is_file()]
    fmt_files = [p for p in FORMATS.iterdir() if p.is_file()]
    print(f'재료 파일 : {len(all_files)}개')
    print(f'  samples 바로 아래 : {len(root_files)}개 · _formats 안 : {len(fmt_files)}개')
    print('_formats 안의 재료 :')
    for p in sorted(FORMATS.glob('sample_*')):
        print('  ' + p.name + ('   ← 오후 03 번' if p.suffix == '.pdf' else ''))


재료 파일 : 30개
  samples 바로 아래 : 24개 · _formats 안 : 6개
_formats 안의 재료 :
  sample_guide.md
  sample_memo.txt
  sample_notice.pptx
  sample_page.html
  sample_scanned.pdf   ← 오후 03 번
  sample_table.xlsx


In [3]:
# 못 읽는 다섯 형식을 한 번에 확인한다.
# 같은 '지원하지 않는 형식' 이지만 이유가 둘로 갈린다 —
# .txt/.md 는 파싱할 것이 없어서, .xlsx/.pptx 는 전용 라이브러리가 필요해서다.
from app.core.exceptions import ValidationFailed
from app.rag.local_parsers import parse_local
today = ['sample_table.xlsx', 'sample_notice.pptx', 'sample_memo.txt', 'sample_guide.md', 'sample_page.html']
for name in today:
    p = FORMATS / name
    try:
        doc = parse_local(p)
        # parse_local 은 세 가지로 끝난다 : 정상 / None(로그만) / ValidationFailed
        if doc is None:
            print(f'{p.name:<20}-> 파서에서 에러가 발생하였습니다. 로그를 확인해주세요')
        else:
            print(f'{p.name:<20}-> 블록 {len(doc.blocks)}개')
    except ValidationFailed as e:
        print(f'{p.name:<20}-> ValidationFailed: {e.message}  | detail={e.detail}')

sample_table.xlsx   -> 블록 1개
sample_notice.pptx  -> 블록 1개
sample_memo.txt     -> ValidationFailed: 지원하지 않는 형식입니다: .txt  | detail=핸들러를 추가하면 지원할 수 있습니다
sample_guide.md     -> ValidationFailed: 지원하지 않는 형식입니다: .md  | detail=핸들러를 추가하면 지원할 수 있습니다
sample_page.html    -> ValidationFailed: 지원하지 않는 형식입니다: .html  | detail=핸들러를 추가하면 지원할 수 있습니다


In [4]:
from app.core.exceptions import ValidationFailed
from app.rag.local_parsers import parse_local

memo = FORMATS / 'sample_memo.txt'
text = memo.read_text(encoding='utf-8')

print(f'[그냥 읽으면]  {len(text)}자')
print(text)
print('[parse_local 에 넣으면]')
try:
    parse_local(memo)
except ValidationFailed as e:
    print(f'ValidationFailed: {e} | detail={e.detail}')

[그냥 읽으면]  71자
출장 정산 안내

9월부터 출장 정산은 그룹웨어에서만 접수합니다.
종이 서류는 받지 않습니다.

문의: 인사총무팀 김지원 과장

[parse_local 에 넣으면]
ValidationFailed: 지원하지 않는 형식입니다: .txt | detail=핸들러를 추가하면 지원할 수 있습니다


In [5]:
# 진입점의 분기만 떼어내서 확인한다. 실제 파싱은 하지 않고 어디로 가는지만 본다.
_PLAIN = {'.txt', '.md'}
_HTML = {'.html', '.htm'}
_LOCAL_ONLY = {'.hwp', '.hwpx'}


def which(name, use_upstage=False):
    ext = pathlib.Path(name).suffix.lower()
    if ext in _PLAIN:
        return '_parse_plain'
    if ext in _HTML:
        return '_parse_html'
    # 순서가 중요하다. 이 조건이 parse_local 보다 앞이라서
    # 스위치를 켜면 docx·pdf·xlsx 가 상용 API 로 넘어간다.
    # 한글 문서(_LOCAL_ONLY)는 켜도 넘어가지 않는다.
    if use_upstage and ext not in _LOCAL_ONLY:
        return 'upstage'
    return 'parse_local'


names = ['sample_memo.txt', 'sample_guide.md', 'sample_page.html', 'sample_table.xlsx', 'sample_notice.pptx', 'DOC-HR-014_국내출장_여비_규정_v2.0.docx', 'DOC-HR-014_국내출장_여비_규정_v2.0.hwpx']
print('[use_upstage=False]')

for n in names:
    print(f'  {n:<40} -> {which(n)}')
print()
print('[use_upstage=True]')

for n in names:
    before, after = (which(n), which(n, use_upstage=True))
    mark = '(바뀐다)' if before != after else '(그대로)'
    print(f'  {n:<40} -> {after:<14} {mark}')

[use_upstage=False]
  sample_memo.txt                          -> _parse_plain
  sample_guide.md                          -> _parse_plain
  sample_page.html                         -> _parse_html
  sample_table.xlsx                        -> parse_local
  sample_notice.pptx                       -> parse_local
  DOC-HR-014_국내출장_여비_규정_v2.0.docx          -> parse_local
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx          -> parse_local

[use_upstage=True]
  sample_memo.txt                          -> _parse_plain   (그대로)
  sample_guide.md                          -> _parse_plain   (그대로)
  sample_page.html                         -> _parse_html    (그대로)
  sample_table.xlsx                        -> upstage        (바뀐다)
  sample_notice.pptx                       -> upstage        (바뀐다)
  DOC-HR-014_국내출장_여비_규정_v2.0.docx          -> upstage        (바뀐다)
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx          -> parse_local    (그대로)


In [6]:
memo = FORMATS / "sample_memo.txt"

# erros="replace" 
# - 깨진 글자 ?로 변환
text = memo.read_text(encoding="utf-8", errors="replace")
# \n 줄바꿈 빈 줄 리스트로 처리
# - 앞 뒤 공백 제거 후 문자열이 있으면 줄 바꿈으로 자른다.
parts = [s.strip() for s in text.split("\n\n") if s.strip()]

for i, s in enumerate(parts, 1):
    print(f"[{i}] {len(s)}자 : {s}")

print(f"블록 {len(parts)}개")

[1] 8자 : 출장 정산 안내
[2] 42자 : 9월부터 출장 정산은 그룹웨어에서만 접수합니다.
종이 서류는 받지 않습니다.
[3] 16자 : 문의: 인사총무팀 김지원 과장
블록 3개


In [7]:
# 같은 함수에 .md 를 넣으면 제목도 블록이 된다.
# '## 1. 신청' 이 8자짜리 블록 하나 — 46번 노트에서 본 '뜻 없는 조각' 문제다.
# 제목을 다음 문단에 붙여야 하는데 아직 안 했다. TODO
guide = FORMATS / "sample_guide.md"

text = guide.read_text(encoding="utf-8", errors="replace")
parts = [s.strip() for s in text.split("\n\n") if s.strip()]

for i, s in enumerate(parts, 1):
    print(f"[{i}] {len(s)}자 : {s}")

print(f"블록 {len(parts)}개")

[1] 13자 : # 출장 신청 빠른 안내
[2] 8자 : ## 1. 신청
[3] 23자 : 출장 3일 전까지 그룹웨어에서 신청합니다.
[4] 8자 : ## 2. 승인
[5] 39자 : 팀장 승인 후 확정됩니다. 5일 초과 출장은 본부장 승인이 필요합니다.
[6] 8자 : ## 3. 정산
[7] 27자 : 복귀 후 7일 이내에 영수증을 첨부해 정산합니다.
블록 7개


In [8]:
# html 페이지에서 태그만 지우기 (나쁜 예시)
# - 정규 표현식 활용
import re

page = FORMATS / "sample_page.html"
html = page.read_text(encoding="utf-8", errors="replace")

# 패턴으로 태그 형태 변경 
flat = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", html)).strip()

print(flat)

blocks = [s.strip() for s in flat.split("\n\n") if s.strip()]
# 하나의 블록으로 합쳐진다.
print(f"블록 {len(blocks)}개")

사내 공지 추석 연휴 근무 안내 연휴 기간 중 당직 근무자는 아래와 같습니다. 날짜 당직자 연락처 10/03 김민준 내선 2201 10/04 이서연 내선 2202
블록 1개


In [9]:
# script, style 태그 묶어서 지우기
raw = re.sub('<(script|style)[^>]*>.*?</\\1>', ' ', html, flags=re.S | re.I)

# 표 모으기
tables = re.findall('<table.*?</table>', raw, flags=re.S | re.I)
# 본문에서 동일한 패턴으로 태그 삭제
body = re.sub('<table.*?</table>', ' ', raw, flags=re.S | re.I)
# 남은 태그를 지우고 공백 제거
body = re.sub('\\s+', ' ', re.sub('<[^>]+>', ' ', body)).strip()

# 떼어둔 표마다 태그를 지워서 표 블록으로 만들기
blocks = [('조항', page.name, body)] if body else []
for i, table in enumerate(tables, 1):
    cleaned = re.sub('\\s+', ' ', re.sub('<[^>]+>', ' ', table)).strip()
    blocks.append(('표', f'{page.name} · 표{i}', cleaned))
# 전체 블록 카운트    
total = sum((len(t) for _, _, t in blocks))
print(f'[좋은 예] 블록 {len(blocks)} · 표 {len(tables)} · 쪽 1 · 글자 {total}자')
# 표가 다 풀려서 출력된다
for n, (kind, locator, t) in enumerate(blocks, 1):
    print(f'  [{n}] {kind}  {locator:<26} {len(t):3d}자  {t[:100]}')
# 추가 고려사항
# - 불필요한 title 태그인 사내 공지가 본문에 들어가고 있음

[좋은 예] 블록 2 · 표 1 · 쪽 1 · 글자 89자
  [1] 조항  sample_page.html            43자  사내 공지 추석 연휴 근무 안내 연휴 기간 중 당직 근무자는 아래와 같습니다.
  [2] 표  sample_page.html · 표1       46자  날짜 당직자 연락처 10/03 김민준 내선 2201 10/04 이서연 내선 2202


In [10]:
import importlib
from app.rag import parser
importlib.reload(parser)
targets = [FORMATS / 'sample_memo.txt', FORMATS / 'sample_guide.md', FORMATS / 'sample_page.html', FORMATS / 'sample_table.xlsx', FORMATS / 'sample_notice.pptx', SAMPLES / 'DOC-HR-014_국내출장_여비_규정_v2.0.docx']
for p in targets:
    try:
        doc = parser.parse(p)
        print(f'{p.name:<40} 블록 {len(doc.blocks):3d}  표 {doc.table_count}  쪽 {doc.page_count}')
    except ValidationFailed as e:
        print(f'{p.name:<40} ValidationFailed: {e}')

sample_memo.txt                          블록   3  표 0  쪽 1
sample_guide.md                          블록   7  표 0  쪽 1
sample_page.html                         블록   2  표 1  쪽 1
sample_table.xlsx                        블록   1  표 1  쪽 1
sample_notice.pptx                       블록   1  표 0  쪽 1
DOC-HR-014_국내출장_여비_규정_v2.0.docx          블록 233  표 7  쪽 1


In [11]:
# workbook : 엑셀 파일
# worksheet : 엑셀 파일 안의 시트
from openpyxl import load_workbook

XLSX = FORMATS / "sample_table.xlsx"

# data_only=True 는 수식이 아니라 계산된 값을 가져온다.
# 끄면 '=SUM(B2:B4)' 가 그대로 나와서 검색에 쓸 수 없다.
wb = load_workbook(str(XLSX), data_only=True)

print("시트 목록 : ", wb.sheetnames)

# 시트 이름으로 시트 하나 추출. 이 이름이 그대로 locator 가 된다.
ws = wb[wb.sheetnames[0]]
print(ws)
print("시트 이름 : ", ws.title)

# print("병합 셀 범위 : ", list(ws.merged_cell_ranges))

# values_only=True 로 받으면 셀 객체가 아니라 값만 튜플로 온다
for row in ws.iter_rows(values_only=True):
    print(row)

시트 목록 :  ['출장비 집계']
<Worksheet "출장비 집계">
시트 이름 :  출장비 집계
('부서', '1분기', '2분기', '3분기')
('인프라사업부 2팀', 4200000, 3800000, 5100000)
('구매팀', 1200000, 1400000, 1100000)
('보안팀', 900000, 1000000, 1300000)


In [12]:
import importlib
from app.rag import local_parsers
importlib.reload(local_parsers)

doc = local_parsers.parse_xlsx(FORMATS / 'sample_table.xlsx')
chars = sum((len(b.text) for b in doc.blocks))
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count} · 글자 {chars}자')
first = doc.blocks[0]
print('  kind    :', first.kind)
print('  locator :', first.locator)
for line in first.text.split('\n')[:3]:
    print(line)

블록 1 · 표 1 · 쪽 1 · 글자 161자
  kind    : 표
  locator : 출장비 집계
| 부서 | 1분기 | 2분기 | 3분기 |
|---|---|---|---|
| 인프라사업부 2팀 | 4200000 | 3800000 | 5100000 |


In [13]:
# 병합 셀이 있는 파일을 직접 만들어 본다. A2:A3 을 병합하면
# 가려진 칸은 값이 None 이 되는데, 그게 파싱에 어떻게 드러나는지 보려는 것이다.
from openpyxl import Workbook

wb = Workbook()
ws = wb.active

ws.title = '부서별 집행액'
ws.append(['구분', '1분기', '2분기'])
ws.append(['인프라사업부', 4200000, 3800000])
ws.append(['인프라사업부', 1200000, 1400000])
ws.append(['구매팀', 900000, 1000000])
ws.merge_cells('A2:A3')
MERGED = SANDBOX / 'merged.xlsx'
wb.save(str(MERGED))

print(f'만든 파일 : {MERGED.relative_to(ROOT)} ({MERGED.stat().st_size:,} 바이트)')

만든 파일 : sandbox\w5\day04\merged.xlsx (5,022 바이트)


In [14]:
# 잘못된 예시
# - 화면에 보이는 도형만 긁는다. 발표자 노트는 slide.shapes 에 없다.
from pptx import Presentation

PPTX = FORMATS / "sample_notice.pptx"
prs = Presentation(str(PPTX))   # PPT 파일 열기

lines = []
for slide in prs.slides:
    for shape in slide.shapes:
        # has_text_frame 으로 거른다. 이미지·선·차트에는 글자 틀이 없어서
        # shape.text 를 바로 읽으면 거기서 예외가 난다.
        if shape.has_text_frame and shape.text.strip():
            lines.append(shape.text.strip())

body = "\n".join(lines)
print(f"슬라이드 : {len(prs.slides)}장, {len(body)}자")
print(body)

슬라이드 : 1장, 70자
출장 여비 규정 개정 안내
일비 25,000 → 30,000원
광역시 숙박비 상한 70,000원 신설
시행 2025-07-01


In [15]:
# 좋은 예시 
from pptx import Presentation

PPTX = FORMATS / "sample_notice.pptx"
prs = Presentation(str(PPTX))   # PPT 파일 열기 

lines = []
for slide in prs.slides:
    for shape in slide.shapes:
        if shape.has_text_frame and shape.text.strip():
            lines.append(shape.text.strip())
    # 슬라이드 기준, 발표자 노트 있는지 체크 
    if slide.has_notes_slide:
        # 발표자 노트의 글자만 긁어오기 
        note = slide.notes_slide.notes_text_frame.text.strip() 
        if note:
            lines.append(f"[발표자 노트] {note}")

body = "\n".join(lines)
print(f"슬라이드 : {len(prs.slides)}장, {len(body)}자")
print(body)

슬라이드 : 1장, 99자
출장 여비 규정 개정 안내
일비 25,000 → 30,000원
광역시 숙박비 상한 70,000원 신설
시행 2025-07-01
[발표자 노트] 인사총무팀 설명용 발표 노트입니다.


In [16]:
# 파싱에서 끝내지 않고 청킹까지 보낸다.
# 블록 수와 청크 수가 크게 어긋나는 앞의 둘을 눈여겨볼 것 — 조가 없는 문서라
# 조 단위 규칙이 걸리지 않아 전부 한 덩이가 된다.
from app.rag import parser
from app.rag.chunker import chunk, summarize
importlib.reload(parser)
FIVE = ['sample_memo.txt', 'sample_guide.md', 'sample_page.html', 'sample_table.xlsx', 'sample_notice.pptx']
for name in FIVE:
    doc = parser.parse(FORMATS / name)
    chunks = chunk(doc)
    print(f'{name:<19} 블록 {len(doc.blocks):>2}  표 {doc.table_count}  →  {summarize(chunks)}')

C:\workspace\hanwha-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sample_memo.txt     블록  3  표 0  →  조항 단위 1 + 표 단위 0 = 1 청크
sample_guide.md     블록  7  표 0  →  조항 단위 1 + 표 단위 0 = 1 청크
sample_page.html    블록  2  표 1  →  조항 단위 1 + 표 단위 1 = 2 청크
sample_table.xlsx   블록  1  표 1  →  조항 단위 0 + 표 단위 1 = 1 청크
sample_notice.pptx  블록  1  표 0  →  조항 단위 1 + 표 단위 0 = 1 청크
